In [1]:
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import numpy as np
from pyscf import gto, scf, fci
from flax import linen as nn
import flax.nnx as nnx
import optax
from tqdm import tqdm
import time 
from functools import partial
from pyscf import gto, scf, fci
from jax import flatten_util
from itertools import combinations
from NES_VMC_H2_631G import get_ccsd_excitations_and_sampler_edges_from_hf,\
SingleStateAnsatz,create_machine,compute_local_energies,forces_expect_hermitian,compute_qgt


/opt/miniconda3/envs/Netket/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


∣NK⟩ Tip: You can use flax.linen, flax.nnx and equinox to define neural networks.

E0 = -1.02613572 Ha  |  激发能: 0.0000 eV
E1 = -0.97892204 Ha  |  激发能: 1.2848 eV
E2 = -0.66776157 Ha  |  激发能: 9.7519 eV
E3 = -0.60817046 Ha  |  激发能: 11.3734 eV


In [2]:
bond_length = 1.8
geometry = [('H', (0., 0., 0.)), ('H', (bond_length, 0., 0.))]
mol = gto.M(atom=geometry, basis='6-31G', verbose=0)
mf = scf.RHF(mol).run(verbose=0)

cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, fcivec = cisolver.kernel()
print("="*60)
print("H₂ FCI 基准能量")
print("="*60)
for i, e in enumerate(E_fcis):
    exc = (e - E_fcis[0]) * 27.2114
    print(f"E{i} = {e:.8f} Ha  |  激发能: {exc:.4f} eV")

# ha = nkx.operator.from_pyscf_molecule(mol)
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=4,
    s=1/2,
    n_fermions_per_spin=(1,1),
)
ha = nkx.operator.from_pyscf_molecule(mol)
Hatree_Fock = hi.all_states()[0]
sampler_edges, singles, doubles = get_ccsd_excitations_and_sampler_edges_from_hf(
    Hatree_Fock
)
print(f'sampler_edges: {sampler_edges}')
g = nk.graph.Graph(edges=sampler_edges)
single_rule = nk.sampler.rules.FermionHopRule(hilbert=hi, graph=g)
sampler = nk.sampler.MetropolisSampler(hi, rule=single_rule, n_chains=100, sweep_size=32)
model = SingleStateAnsatz(n_spin_orbitals=hi.size,hidden_dim=hi.size*2,rngs=nnx.Rngs(12))
vstate = nk.vqs.MCState(sampler, model, n_samples=1008)

optimizer = nk.optimizer.Sgd(learning_rate=0.1)

gs = nk.driver.VMC(
    ha,
    optimizer,
    variational_state=vstate,
    preconditioner=nk.optimizer.SR(diag_shift=0.1,holomorphic=True),
)


H₂ FCI 基准能量
E0 = -1.02613572 Ha  |  激发能: 0.0000 eV
E1 = -0.97892204 Ha  |  激发能: 1.2848 eV
E2 = -0.66776157 Ha  |  激发能: 9.7519 eV
E3 = -0.60817046 Ha  |  激发能: 11.3734 eV
sampler_edges: [(3, 0), (3, 1), (3, 2), (7, 4), (7, 5), (7, 6)]


/opt/miniconda3/envs/Netket/lib/python3.11/site-packages/netket/vqs/mc/mc_state/state.py:314: UserWarning: n_samples=1008 (1008 per JAX device) does not divide n_chains=100, increased to 1100 (1100 per JAX device)
  self.n_samples = n_samples


In [3]:
gs = nk.driver.VMC(
    ha,
    optimizer,
    variational_state=vstate,
)
gs.run(400)

100%|██████████| 400/400 [00:17<00:00, 22.49it/s, Energy=-1.0192+0.0007j ± 0.0038 [σ²=2.3e-03, R̂=1.269]]                                                               


()